<a href="https://colab.research.google.com/github/deepavasanthkumar/modern_lakehouse_datafusion/blob/main/datafusion_datalake_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## DataLake using DataFusion - Visualization !

In [1]:

!pip install datafusion pyarrow pandas plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.1/33.1 MB 41.2 MB/s eta 0:00:00


In [4]:
from datafusion import SessionContext
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
import os
import plotly.express as px

## Step 1 — Generate a Large Synthetic DatasetWe simulate an e-commerce dataset with millions of rows.

In [7]:
rows = 2_000_000
countries = ["US","India","UK","Germany","Canada"]
products = ["Laptop","Phone","Tablet","Monitor","Headphones"]
df = pd.DataFrame({    "order_id": np.arange(rows),    "country": np.random.choice(countries, rows),    "product": np.random.choice(products, rows),    "price": np.random.randint(50,1000,rows),    "quantity": np.random.randint(1,5,rows)})
df["sales"] = df["price"] * df["quantity"]
df.head()

,order_id,country,product,price,quantity,sales
0,0,Germany,Phone,216,2,432
1,1,Canada,Tablet,554,2,1108
2,2,UK,Tablet,367,2,734
3,3,US,Monitor,361,3,1083
4,4,US,Headphones,103,2,206


## Step 2 — Write Data Lake Parquet FilesWe simulate a **data lake layout** with multiple Parquet files.

In [8]:
os.makedirs("lake/orders", exist_ok=True)
chunk_size = 200000
for i in range(0, len(df), chunk_size):
  chunk = df.iloc[i:i+chunk_size]
  table = pa.Table.from_pandas(chunk)
  pq.write_table(table, f"lake/orders/orders_{i}.parquet")
print("Parquet files written.")

Parquet files written.


## Step 3 — Create DataFusion Session

In [9]:
ctx = SessionContext()

## Step 4 — Register Data Lake DirectoryDataFusion can query multiple Parquet files as one dataset.

In [10]:
ctx.register_parquet("orders", "lake/orders/*.parquet")

## Step 5 — Explore Dataset

In [11]:
ctx.sql("SELECT COUNT(*) as total_rows FROM orders").to_pandas()

,total_rows
0,2000000


## Step 6 — Country Sales Analytics

In [12]:
country_sales = ctx.sql("""SELECT country,SUM(sales) as total_sales FROM orders GROUP BY country""").to_pandas()
country_sales

,country,total_sales
0,Germany,524357319
1,UK,524845448
2,Canada,524998263
3,India,525300445
4,US,524505274


In [13]:
fig = px.bar(    country_sales,    x="country",    y="total_sales",    title="Total Sales by Country")
fig.show()

## Step 7 — Product Analytics

In [14]:
product_sales = ctx.sql("""SELECT product,SUM(sales) as total_sales,COUNT(*) as orders FROM orders GROUP BY product""").to_pandas()
product_sales

,product,total_sales,orders
0,Headphones,524925705,399737
1,Laptop,525728918,400577
2,Monitor,524455461,400017
3,Tablet,524802976,399932
4,Phone,524093689,399737


In [15]:
fig = px.bar(    product_sales,    x="product",    y="total_sales",    title="Sales by Product")
fig.show()

## Step 8 — Advanced Analytical QueryExample of a more complex SQL query.

In [16]:
analytics = ctx.sql("""SELECT country,product,SUM(sales) as total_sales,AVG(price) as avg_price,COUNT(*) as orders FROM orders GROUP BY country, product""").to_pandas()
analytics.head()

,country,product,total_sales,avg_price,orders
0,Germany,Laptop,105837017,525.057445,80686
1,UK,Tablet,104933349,524.133927,80096
2,UK,Phone,104726282,523.184553,80069
3,UK,Laptop,104866854,524.765778,79860
4,Germany,Tablet,104137718,522.710168,79767


In [17]:
fig = px.scatter(    analytics,    x="orders",    y="total_sales",    color="country",    size="avg_price",    title="Orders vs Sales by Country/Product")
fig.show()

## Step 9 — Query Execution Plan


In [18]:
plan = ctx.sql("""SELECT country, SUM(sales)FROM orders GROUP BY country""")
print(plan.explain())

DataFrame()
+---------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| plan_type     | plan                                                                                                                                                                                                                                                                                                             

This notebook demonstrates a lightweight analytics stack:

✅Parquet Data Lake        
✅ Apache DataFusion SQL Engine        
✅Pandas DataFrame        
✅Interactive Analytics (Plotly)

This architecture is conceptually similar to:- Spark SQL- Trino- DuckDB
